# Experimentos

## Importando requisitos

In [ ]:
import pandas
import os
import numpy

## Abrindo a planilha

In [ ]:
planilha = pandas.read_csv('../Dataset/DataSummary.csv')
planilha

## Filtrando na planilha os datasets selecionados

In [ ]:
planilha = planilha[planilha['ID'].between(97, 128)]
planilha

## Realizando experimentos no dataset da primeira linha da tabela

Obtendo o caminho do dataset de exemplo

In [ ]:
dataset_exemplo = planilha.iloc[0]['Name']
dataset_exemplo

In [ ]:
dataset_dirname = os.path.join('../Dataset/UCRArchive_2018', dataset_exemplo)
dataset_dirname

In [ ]:
caminho_arquivo_treino = os.path.join(dataset_dirname, '{}_TRAIN.tsv'.format(dataset_exemplo))
caminho_arquivo_treino

In [ ]:
caminho_arquivo_teste = os.path.join(dataset_dirname, '{}_TEST.tsv'.format((dataset_exemplo)))
caminho_arquivo_teste

Abrindo datasets

In [ ]:
dataset_exemplo_treino = pandas.read_csv(caminho_arquivo_treino, sep='\t', header=None)
dataset_exemplo_treino

In [ ]:
dataset_exemplo_teste = pandas.read_csv(caminho_arquivo_teste, sep='\t', header=None)
dataset_exemplo_teste

### Formatando Dataset

In [ ]:
def formatar_dataset(df: pandas.DataFrame) -> pandas.DataFrame:
    """
    Realiza a formatação do dataset.
    :param df: Dataset a ser formatado.
    :return: Dataset formatado.
    """
    classe = df.iloc[:, 0]
    series = df.iloc[:, 1:]

    df_novo = pandas.DataFrame({
        'classe': classe,
        'SérieTemporal': list(series.to_numpy())
    })

    return df_novo

In [ ]:
dataset_exemplo_treino = formatar_dataset(dataset_exemplo_treino)
dataset_exemplo_treino

In [ ]:
dataset_exemplo_teste = formatar_dataset(dataset_exemplo_teste)
dataset_exemplo_teste

### Aplicando algoritmos nos datasets

In [ ]:
from app.model.DynamicTimeWarping import DynamicTimeWarping
from app.model.DerivativeDynamicTimeWarping import DerivativeDynamicTimeWarping
from app.model.LongestCommonSubsequence import LongestCommonSubsequence
from app.model.SoftDynamicTimeWarping import SoftDynamicTimeWarping

def aplicar_algoritmos_series_temporais(dataset: pandas.DataFrame, indice_serie_referencia: int = 0) -> pandas.DataFrame:
    """
    Aplica algoritmos às séries temporais.
    :param dataset: Dataset.
    :param indice_serie_referencia: Índice da série temporal de referência.
    :return: Dataset com as distâncias das séries temporais.
    """
    # Instanciando algoritmos
    dtw = DynamicTimeWarping()
    ddtw = DerivativeDynamicTimeWarping()
    lcs = LongestCommonSubsequence()
    soft_dtw = SoftDynamicTimeWarping()

    # Definindo série de referência
    serie_referencia = dataset.iloc[indice_serie_referencia]["SérieTemporal"]

    # Executando algoritmos e adicionando ao Dataset
    print('Executando Dynamic Time Warping')
    dataset_exemplo_treino['dtw'] = dataset_exemplo_treino['SérieTemporal'].apply(
        lambda s: dtw.obter_distancia(s, serie_referencia)
    )
    print('Finalizado o Dynamic Time Warping')

    print('Executando o Derivative Dynamic Time Warping')
    dataset_exemplo_treino['ddtw'] = dataset_exemplo_treino['SérieTemporal'].apply(
        lambda s: ddtw.obter_distancia(s, serie_referencia)
    )
    print('Finalizado o Derivative Dynamic Time Warping')

    print('Executando o Longest Common Subsequence')
    dataset_exemplo_treino['lcs'] = dataset_exemplo_treino['SérieTemporal'].apply(
        lambda s: lcs.lcs(s, serie_referencia)
    )
    print('Finalizado o Longest Common Subsequence')

    print('Executando o Soft Dynamic Time Warping')
    dataset_exemplo_treino['soft-dtw'] = dataset_exemplo_treino['SérieTemporal'].apply(
        lambda s: soft_dtw.obter_distancia(s, serie_referencia)
    )
    print('Finalizado o Soft Dynamic Time Warping')

    return dataset

Função auxiliar

In [ ]:
import os
import pathlib
import json

def salvar_como_json(dados: object, filepath: str):
    """
    Salva um objeto como JSON.
    :param dados: Objeto a ser salvo
    :param filepath: Caminho do arquivo onde será salvo o objeto.
    :return:
    """

    # Criando o diretório pai, caso ele não exista
    os.makedirs(pathlib.Path(filepath).parent, exist_ok=True)

    with open(filepath, 'w', encoding='utf-8') as arquivo:
        json.dump(dados, arquivo, indent=4, ensure_ascii=False)

In [ ]:
def carregar_arquivo_json(filepath: str) -> object:
    """
    Carrega um arquivo JSON.
    :param filepath: Caminho do arquivo onde será lido o objeto.
    :return: Objeto carregado a partir do arquivo JSON.
    """
    with open(filepath, 'r', encoding='utf-8') as arquivo:
        return json.load(arquivo)

In [ ]:
dataset_exemplo_treino = aplicar_algoritmos_series_temporais(dataset_exemplo_treino)

In [ ]:
dataset_exemplo_teste = aplicar_algoritmos_series_temporais(dataset_exemplo_teste)

Salvando os datasets como JSON

In [ ]:
salvar_como_json(filepath='../Dataset/processed/dataset_exemplo_treino.json', dados=dataset_exemplo_treino)
salvar_como_json(filepath='../Dataset/processed/dataset_exemplo_teste.json', dados=dataset_exemplo_teste)

### Criando conjuntos de teste e treino

Carregando os dados a partir do arquivo JSON

In [ ]:
dataset_exemplo_treino = carregar_arquivo_json(filepath='../Dataset/processed/dataset_exemplo_treino.json')
dataset_exemplo_teste = carregar_arquivo_json(filepath='../Dataset/processed/dataset_exemplo_teste.json')

In [ ]:
dataset_exemplo_treino

In [ ]:
dataset_exemplo_teste

Obtendo conjuntos de treino e teste: x e y

In [ ]:
def separar_x_y(dataset: pandas.DataFrame) -> tuple[list, list]:
    """
    Separa o conjunto de treino do conjunto de teste
    :param dataset: Dataset de entrada.
    :return: Tupla com os cunjuntos de treino e teste.
    """
    x = []
    for linha in dataset.iloc[:, 1:].values:
        x.append(linha)

    y = []
    for linha in dataset.iloc[:, 0].values:
        y.append(linha)

    return x, y

In [ ]:
x_train, y_train = separar_x_y(dataset_exemplo_treino)

In [ ]:
x_train

In [ ]:
y_train

In [ ]:
x_test, y_test = separar_x_y(dataset_exemplo_teste)

### Treinando modelo

Normalizando os dados

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.fit_transform(x_test)

Criando e treinando modelo

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
knn = KNeighborsClassifier()
knn.fit(x_train, y_train)

Realizando predições

In [ ]:
y_pred = knn.predict(x_test)
y_pred

Avaliando modelo

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
acuracia = accuracy_score(y_test, y_pred)
acuracia